# Module 04 — ML Theory & Evaluation
## 04-02: Cross-Validation & Hyperparameter Tuning

**Objective:** Implement k-fold, stratified k-fold, leave-one-out, and nested cross-validation from scratch; build grid search and random search; and use learning curves to diagnose bias vs variance.

**Prerequisites:** 4-01 (Evaluation Metrics)

## Part 0 — Setup & Prerequisites

This notebook owns **k-fold CV, stratified k-fold, leave-one-out CV, nested CV, grid search, random search, and learning curves**. Proper cross-validation is the foundation of every honest model evaluation in this course — it prevents the "test set contamination" that arises when hyperparameters are tuned on the same data used to estimate generalization. Nested CV is the gold standard: the inner loop tunes, the outer loop estimates.

**Prerequisites:** 4-01 (Evaluation Metrics)

> **Note (Category C):** This notebook focuses on the evaluation methodology. We use Logistic Regression and Random Forest as vehicles to demonstrate CV behaviour; their internals are covered in Module 2.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import warnings
import random
import time
import itertools
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn import model_selection as skms

import torch
from typing import Any

import copy
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
print(f'Python : {sys.version.split()[0]}')
print(f'NumPy  : {np.__version__}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Constants ─────────────────────────────────────────────────────────────────
K_FOLDS: int = 5            # Standard k for k-fold CV
K_OUTER: int = 5            # Outer folds for nested CV
K_INNER: int = 3            # Inner folds for nested CV (hyperparameter selection)
N_ITER_RANDOM: int = 40     # Random search iterations
LR_MAX_ITER: int = 500      # LogisticRegression solver iterations
RF_N_EST: int = 50          # Random Forest estimators (kept small for speed)
N_TRAIN_SIZES: int = 8      # Points on the learning curve

print(f'k-fold K     : {K_FOLDS}')
print(f'Nested CV    : {K_OUTER}-outer / {K_INNER}-inner')
print(f'Random search: {N_ITER_RANDOM} iterations')

### Data Loading & Exploration

We use the **sklearn Digits** dataset throughout: 1 797 samples, 10 classes, 64 features (8×8 pixel values). It is small enough for nested CV to run in a few seconds, yet rich enough to show meaningful bias-variance tradeoffs.

In [ ]:
# ── Load Digits ───────────────────────────────────────────────────────────────
digits = load_digits()
X_all: np.ndarray = digits.data    # (1797, 64)
y_all: np.ndarray = digits.target  # (1797,)  0..9
n_classes: int = len(digits.target_names)
n_samples: int = X_all.shape[0]
n_features: int = X_all.shape[1]

print(f'Digits dataset: {n_samples} samples, {n_features} features, {n_classes} classes')

# Class counts — confirms near-perfect balance
unique, counts = np.unique(y_all, return_counts=True)
print('Class counts:', dict(zip(unique.tolist(), counts.tolist())))

# Hold out a final test set — this is NEVER used for any CV or tuning decision
rng_split = np.random.RandomState(SEED)
test_idx: np.ndarray = rng_split.choice(n_samples, size=int(0.2 * n_samples), replace=False)
train_idx: np.ndarray = np.setdiff1d(np.arange(n_samples), test_idx)

X_dev: np.ndarray = X_all[train_idx]   # Development set (CV is run only on this)
y_dev: np.ndarray = y_all[train_idx]
X_test: np.ndarray = X_all[test_idx]   # Final test set (locked away)
y_test: np.ndarray = y_all[test_idx]

print(f'\nDevelopment set : {X_dev.shape[0]} samples')
print(f'Final test set  : {X_test.shape[0]} samples  (held out, not used in CV)')

# Quick visualisation: sample digits
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for k, ax in enumerate(axes.flatten()):
    sample_idx = np.where(y_dev == k)[0][0]
    ax.imshow(X_dev[sample_idx].reshape(8, 8), cmap='grey_r', interpolation='nearest')
    ax.set_title(f'Digit {k}', fontsize=9)
    ax.axis('off')
plt.suptitle('One Sample per Class (Development Set)', fontsize=11)
plt.tight_layout()
plt.savefig('04_02_digits_samples.png', dpi=100, bbox_inches='tight')
plt.show()

---
## Part 1 — CV Strategies & Search Algorithms from Scratch

### 1.1 K-Fold and Stratified K-Fold

**K-fold CV** partitions the dataset into $k$ non-overlapping folds. Each fold serves as the validation set exactly once, and the model is trained on the remaining $k-1$ folds. The $k$ validation scores are averaged to estimate generalization performance.

**Stratified k-fold** preserves the class distribution in every fold — critical when classes are imbalanced. For a balanced dataset like Digits the difference is minor, but for imbalanced tasks it prevents degenerate folds.

In [ ]:
def k_fold_indices(
    n_samples: int,
    k: int,
    shuffle: bool = True,
    random_state: int = SEED,
) -> list[tuple[np.ndarray, np.ndarray]]:
    '''Generate train/validation index pairs for k-fold cross-validation.

    Args:
        n_samples: Total number of samples.
        k: Number of folds.
        shuffle: If True, shuffle indices before splitting.
        random_state: Seed for the shuffle RNG.

    Returns:
        List of k (train_indices, val_indices) tuples.
    '''
    indices: np.ndarray = np.arange(n_samples)
    if shuffle:
        rng = np.random.RandomState(random_state)
        rng.shuffle(indices)

    fold_sizes: np.ndarray = np.full(k, n_samples // k, dtype=int)
    fold_sizes[: n_samples % k] += 1   # distribute remainder

    splits: list = []
    current: int = 0
    for fold_size in fold_sizes:
        val_idx: np.ndarray = indices[current: current + fold_size]
        train_idx: np.ndarray = np.concatenate(
            [indices[:current], indices[current + fold_size:]]
        )
        splits.append((train_idx, val_idx))
        current += fold_size

    return splits


def stratified_k_fold_indices(
    y: np.ndarray,
    k: int,
    shuffle: bool = True,
    random_state: int = SEED,
) -> list[tuple[np.ndarray, np.ndarray]]:
    '''Generate train/validation index pairs for stratified k-fold CV.

    Each fold preserves the class proportion of the full dataset, which
    prevents degenerate folds on imbalanced class distributions.

    Args:
        y: Integer class label array of length n_samples.
        k: Number of folds.
        shuffle: If True, shuffle within each class before assigning to folds.
        random_state: Seed for the shuffle RNG.

    Returns:
        List of k (train_indices, val_indices) tuples.
    '''
    rng = np.random.RandomState(random_state)
    classes: np.ndarray = np.unique(y)

    # Assign each class sample to folds in round-robin order
    fold_assignment: np.ndarray = np.empty(len(y), dtype=int)
    for cls in classes:
        cls_idx: np.ndarray = np.where(y == cls)[0]
        if shuffle:
            rng.shuffle(cls_idx)
        fold_assignment[cls_idx] = np.arange(len(cls_idx)) % k

    all_indices: np.ndarray = np.arange(len(y))
    splits: list = []
    for fold in range(k):
        val_idx: np.ndarray = all_indices[fold_assignment == fold]
        train_idx: np.ndarray = all_indices[fold_assignment != fold]
        splits.append((train_idx, val_idx))

    return splits


# Verify fold sizes and class distribution
folds_kf = k_fold_indices(len(y_dev), K_FOLDS)
folds_skf = stratified_k_fold_indices(y_dev, K_FOLDS)

print(f'K-Fold (k={K_FOLDS}) fold sizes:')
for i, (tr, vl) in enumerate(folds_kf):
    print(f'  Fold {i+1}: train={len(tr)}, val={len(vl)}')

print(f'\nStratified K-Fold — val class distribution per fold:')
for i, (_, vl) in enumerate(folds_skf):
    cls_counts = {int(c): int((y_dev[vl] == c).sum()) for c in range(n_classes)}
    min_count = min(cls_counts.values())
    max_count = max(cls_counts.values())
    print(f'  Fold {i+1}: val={len(vl)}, '
          f'per-class range [{min_count}, {max_count}]')

### 1.2 Leave-One-Out Cross-Validation

LOO-CV is a special case of k-fold where $k = n$. Each sample is the validation set exactly once. It is **nearly unbiased** (trained on $n-1$ samples ≈ trained on all $n$ samples) but has **high variance** — each fold produces a single binary 0/1 score. LOO is expensive ($n$ model fits) but useful when data is very scarce.

In [ ]:
def leave_one_out_indices(
    n_samples: int,
) -> list[tuple[np.ndarray, np.ndarray]]:
    '''Generate train/validation pairs for leave-one-out cross-validation.

    Each fold uses a single sample as the validation set, so n_samples
    folds are produced in total.

    Args:
        n_samples: Total number of samples.

    Returns:
        List of n_samples (train_indices, val_indices) tuples.
    '''
    all_idx: np.ndarray = np.arange(n_samples)
    splits: list = []
    for i in range(n_samples):
        val_idx: np.ndarray = np.array([i])
        train_idx: np.ndarray = np.delete(all_idx, i)
        splits.append((train_idx, val_idx))
    return splits


# Demonstrate on a small subset (LOO on full 1437 would be slow)
n_loo_demo: int = 60
X_loo_demo: np.ndarray = X_dev[:n_loo_demo]
y_loo_demo: np.ndarray = y_dev[:n_loo_demo]

loo_splits = leave_one_out_indices(n_loo_demo)
print(f'LOO on {n_loo_demo} samples: {len(loo_splits)} folds generated.')
print(f'Each fold — train: {len(loo_splits[0][0])}, val: {len(loo_splits[0][1])}')

### 1.3 Cross-Validation Scoring Function

The `cross_validate_scratch` function accepts any sklearn-compatible estimator and a list of fold splits, fits the model on each training fold, and collects the validation scores. It also records fit and score times per fold.

In [ ]:
def cross_validate_scratch(
    estimator: Any,
    X: np.ndarray,
    y: np.ndarray,
    splits: list[tuple[np.ndarray, np.ndarray]],
    scoring: str = 'accuracy',
    scale: bool = True,
) -> dict:
    '''Run cross-validation and collect per-fold metrics.

    Fits a fresh copy of the estimator on each training fold, evaluates on
    the validation fold, and returns all per-fold scores.

    Args:
        estimator: sklearn-compatible estimator with fit() and predict().
        X: Feature matrix of shape (n_samples, n_features).
        y: Label array of shape (n_samples,).
        splits: List of (train_idx, val_idx) tuples from a splitter function.
        scoring: Metric to compute. Supports 'accuracy' and 'f1_macro'.
        scale: If True, apply StandardScaler fit on training fold, transform both.

    Returns:
        Dictionary with keys 'val_scores', 'train_scores', 'fit_times',
        'score_times', each a list of length k.
    '''

    val_scores: list = []
    train_scores: list = []
    fit_times: list = []
    score_times: list = []

    for train_idx, val_idx in splits:
        X_tr_fold: np.ndarray = X[train_idx]
        y_tr_fold: np.ndarray = y[train_idx]
        X_vl_fold: np.ndarray = X[val_idx]
        y_vl_fold: np.ndarray = y[val_idx]

        if scale:
            sc = StandardScaler()
            X_tr_fold = sc.fit_transform(X_tr_fold)
            X_vl_fold = sc.transform(X_vl_fold)

        model = copy.deepcopy(estimator)

        t0 = time.perf_counter()
        model.fit(X_tr_fold, y_tr_fold)
        fit_times.append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        y_pred_tr: np.ndarray = model.predict(X_tr_fold)
        y_pred_vl: np.ndarray = model.predict(X_vl_fold)
        score_times.append(time.perf_counter() - t0)

        if scoring == 'accuracy':
            val_scores.append(accuracy_score(y_vl_fold, y_pred_vl))
            train_scores.append(accuracy_score(y_tr_fold, y_pred_tr))
        elif scoring == 'f1_macro':
            val_scores.append(
                f1_score(y_vl_fold, y_pred_vl, average='macro', zero_division=0))
            train_scores.append(
                f1_score(y_tr_fold, y_pred_tr, average='macro', zero_division=0))
        else:
            raise ValueError(f'Unknown scoring: {scoring!r}')

    return {
        'val_scores': val_scores,
        'train_scores': train_scores,
        'fit_times': fit_times,
        'score_times': score_times,
    }



### 1.4 Learning Curves

A **learning curve** plots train and validation score as a function of training set size. It is the primary tool for diagnosing **bias vs variance**:

- **High bias (underfitting):** Both train and val scores are low; they converge to the same low value. Adding more data barely helps.
- **High variance (overfitting):** Train score is high but val score is much lower; there is a large gap. Adding more data helps close the gap.

We implement learning curves from scratch without using `sklearn.model_selection.learning_curve`.

In [ ]:
def learning_curve_scratch(
    estimator: Any,
    X: np.ndarray,
    y: np.ndarray,
    train_sizes: np.ndarray,
    k: int = 5,
    scoring: str = 'accuracy',
    scale: bool = True,
    random_state: int = SEED,
) -> dict:
    '''Compute learning curve: train/val score vs training set size.

    For each training set size, runs k-fold CV on a stratified subsample
    and records mean/std of train and validation scores.

    Args:
        estimator: sklearn-compatible estimator.
        X: Feature matrix of shape (n_samples, n_features).
        y: Label array of shape (n_samples,).
        train_sizes: 1-D array of absolute training set sizes to evaluate.
        k: Number of CV folds at each training set size.
        scoring: Metric string, passed to cross_validate_scratch.
        scale: If True, StandardScale inside each fold.
        random_state: RNG seed for reproducible fold splits.

    Returns:
        Dictionary with keys 'train_sizes', 'train_mean', 'train_std',
        'val_mean', 'val_std'. Each value is a 1-D array of length
        len(train_sizes).
    '''
    train_means: list = []
    train_stds: list = []
    val_means: list = []
    val_stds: list = []

    for size in train_sizes:
        # Subsample `size` examples in a stratified manner
        rng = np.random.RandomState(random_state)
        classes = np.unique(y)
        per_class: int = max(k, size // len(classes))  # at least k per class
        chosen_idx_list: list = []
        for cls in classes:
            cls_idx = np.where(y == cls)[0]
            n_take = min(per_class, len(cls_idx))
            chosen = rng.choice(cls_idx, size=n_take, replace=False)
            chosen_idx_list.append(chosen)
        chosen_idx: np.ndarray = np.concatenate(chosen_idx_list)
        # Trim to exactly `size` samples if over
        if len(chosen_idx) > size:
            rng.shuffle(chosen_idx)
            chosen_idx = chosen_idx[:size]

        X_sub: np.ndarray = X[chosen_idx]
        y_sub: np.ndarray = y[chosen_idx]

        splits_sub = stratified_k_fold_indices(y_sub, k, shuffle=True,
                                                random_state=random_state)
        cv_result = cross_validate_scratch(
            estimator, X_sub, y_sub, splits_sub,
            scoring=scoring, scale=scale,
        )
        train_means.append(float(np.mean(cv_result['train_scores'])))
        train_stds.append(float(np.std(cv_result['train_scores'])))
        val_means.append(float(np.mean(cv_result['val_scores'])))
        val_stds.append(float(np.std(cv_result['val_scores'])))

    return {
        'train_sizes': np.array(train_sizes, dtype=int),
        'train_mean': np.array(train_means),
        'train_std': np.array(train_stds),
        'val_mean': np.array(val_means),
        'val_std': np.array(val_stds),
    }


def plot_learning_curve(
    lc: dict,
    title: str = 'Learning Curve',
    ax: plt.Axes | None = None,
    color_train: str = 'steelblue',
    color_val: str = 'salmon',
) -> plt.Axes:
    '''Plot a learning curve with shaded standard-deviation bands.

    Args:
        lc: Dictionary returned by learning_curve_scratch.
        title: Plot title.
        ax: Axes to draw on; creates a new figure if None.
        color_train: Line/band color for training scores.
        color_val: Line/band color for validation scores.

    Returns:
        The Axes object with the learning curve drawn.
    '''
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4))

    sizes = lc['train_sizes']
    ax.fill_between(
        sizes,
        lc['train_mean'] - lc['train_std'],
        lc['train_mean'] + lc['train_std'],
        alpha=0.15, color=color_train,
    )
    ax.fill_between(
        sizes,
        lc['val_mean'] - lc['val_std'],
        lc['val_mean'] + lc['val_std'],
        alpha=0.15, color=color_val,
    )
    ax.plot(sizes, lc['train_mean'], 'o-', color=color_train, lw=2,
            label='Train score')
    ax.plot(sizes, lc['val_mean'], 's-', color=color_val, lw=2,
            label='Val score')
    ax.set_xlabel('Training set size')
    ax.set_ylabel('Accuracy')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)
    return ax



### 1.5 Grid Search and Random Search

**Grid search** exhaustively evaluates every combination in a hyperparameter grid. With $p$ parameters each taking $n_i$ values, the total number of evaluations is $\prod_i n_i$, which grows exponentially.

**Random search** (Bergstra & Bengio, 2012) instead samples parameter combinations randomly. For a fixed budget of $T$ evaluations, random search often finds better configurations than grid search because it covers more of the *important* dimensions — the insight is that most hyperparameter spaces have low effective dimensionality.

In [ ]:
class GridSearchCV:
    '''Exhaustive grid search with k-fold cross-validation.

    Evaluates every combination of hyperparameters in the provided grid
    using stratified k-fold CV on the provided data, then exposes the
    best configuration and its score.

    Attributes:
        estimator: The base estimator to clone for each configuration.
        param_grid: Dict mapping parameter name to list of candidate values.
        k: Number of CV folds.
        scoring: Metric string ('accuracy' or 'f1_macro').
        scale: Whether to StandardScale inside each fold.
        best_params_: Best parameter dict after fitting.
        best_score_: Best mean CV score after fitting.
        cv_results_: DataFrame of all evaluated configurations and scores.
    '''

    def __init__(
        self,
        estimator: Any,
        param_grid: dict,
        k: int = 5,
        scoring: str = 'accuracy',
        scale: bool = True,
        random_state: int = SEED,
    ) -> None:
        '''Initialise GridSearchCV.

        Args:
            estimator: sklearn-compatible base estimator.
            param_grid: Dict of {param_name: [val1, val2, ...]}.
            k: Number of CV folds.
            scoring: Metric to optimise ('accuracy' or 'f1_macro').
            scale: If True, apply StandardScaler inside each CV fold.
            random_state: Seed for stratified fold generation.
        '''
        self.estimator = estimator
        self.param_grid: dict = param_grid
        self.k: int = k
        self.scoring: str = scoring
        self.scale: bool = scale
        self.random_state: int = random_state
        self.best_params_: dict | None = None
        self.best_score_: float = -float('inf')
        self.cv_results_: pd.DataFrame | None = None

    def _iter_param_combinations(self) -> list[dict]:
        '''Return all combinations of parameters in param_grid.

        Returns:
            List of dicts, each mapping param name to a specific value.
        '''
        keys = list(self.param_grid.keys())
        value_lists = [self.param_grid[k] for k in keys]
        combos: list = []
        for combo in itertools.product(*value_lists):
            combos.append(dict(zip(keys, combo)))
        return combos

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'GridSearchCV':
        '''Run grid search with CV on (X, y).

        Args:
            X: Feature matrix of shape (n_samples, n_features).
            y: Label array of shape (n_samples,).

        Returns:
            Self (fitted).
        '''
        combos = self._iter_param_combinations()
        splits = stratified_k_fold_indices(
            y, self.k, shuffle=True, random_state=self.random_state)

        records: list = []
        for params in combos:
            model = copy.deepcopy(self.estimator)
            model.set_params(**params)
            cv = cross_validate_scratch(
                model, X, y, splits,
                scoring=self.scoring, scale=self.scale,
            )
            mean_score = float(np.mean(cv['val_scores']))
            std_score = float(np.std(cv['val_scores']))
            record = dict(params)
            record['mean_val_score'] = mean_score
            record['std_val_score'] = std_score
            records.append(record)

            if mean_score > self.best_score_:
                self.best_score_ = mean_score
                self.best_params_ = dict(params)

        self.cv_results_ = pd.DataFrame(records).sort_values(
            'mean_val_score', ascending=False).reset_index(drop=True)
        return self



In [ ]:
class RandomSearchCV:
    '''Random hyperparameter search with k-fold cross-validation.

    Samples parameter combinations randomly from either discrete lists
    or continuous ranges (specified as (low, high, 'log') tuples).

    Attributes:
        estimator: Base estimator.
        param_distributions: Dict of {param_name: list | (low, high, scale)}.
        n_iter: Number of random configurations to try.
        k: Number of CV folds.
        scoring: Metric string.
        scale: Whether to StandardScale inside each fold.
        best_params_: Best parameter dict after fitting.
        best_score_: Best mean CV score after fitting.
        cv_results_: DataFrame of all evaluated configurations and scores.
    '''

    def __init__(
        self,
        estimator: Any,
        param_distributions: dict,
        n_iter: int = 20,
        k: int = 5,
        scoring: str = 'accuracy',
        scale: bool = True,
        random_state: int = SEED,
    ) -> None:
        '''Initialise RandomSearchCV.

        Args:
            estimator: sklearn-compatible base estimator.
            param_distributions: Dict mapping param name to:
                - A list of discrete values to sample from, or
                - A tuple (low, high, scale) where scale is 'linear' or 'log'
                  for continuous sampling.
            n_iter: Number of random configurations to evaluate.
            k: Number of CV folds.
            scoring: Metric to optimise ('accuracy' or 'f1_macro').
            scale: If True, apply StandardScaler inside each CV fold.
            random_state: Seed for random sampling.
        '''
        self.estimator = estimator
        self.param_distributions: dict = param_distributions
        self.n_iter: int = n_iter
        self.k: int = k
        self.scoring: str = scoring
        self.scale: bool = scale
        self.random_state: int = random_state
        self.best_params_: dict | None = None
        self.best_score_: float = -float('inf')
        self.cv_results_: pd.DataFrame | None = None

    def _sample_params(self, rng: np.random.RandomState) -> dict:
        '''Sample one parameter configuration.

        Args:
            rng: RandomState used for sampling.

        Returns:
            Dict of sampled parameter values.
        '''
        params: dict = {}
        for name, dist in self.param_distributions.items():
            if isinstance(dist, list):
                params[name] = rng.choice(dist)
            elif isinstance(dist, tuple) and len(dist) == 3:
                low, high, scale = dist
                if scale == 'log':
                    params[name] = float(np.exp(
                        rng.uniform(np.log(low), np.log(high))))
                else:
                    params[name] = float(rng.uniform(low, high))
            else:
                raise ValueError(f'Unknown distribution spec for {name!r}: {dist}')
        return params

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'RandomSearchCV':
        '''Run random search with CV on (X, y).

        Args:
            X: Feature matrix of shape (n_samples, n_features).
            y: Label array of shape (n_samples,).

        Returns:
            Self (fitted).
        '''
        rng = np.random.RandomState(self.random_state)
        splits = stratified_k_fold_indices(
            y, self.k, shuffle=True, random_state=self.random_state)

        records: list = []
        for _ in range(self.n_iter):
            params = self._sample_params(rng)
            model = copy.deepcopy(self.estimator)
            model.set_params(**params)
            cv = cross_validate_scratch(
                model, X, y, splits,
                scoring=self.scoring, scale=self.scale,
            )
            mean_score = float(np.mean(cv['val_scores']))
            std_score = float(np.std(cv['val_scores']))
            record = dict(params)
            record['mean_val_score'] = mean_score
            record['std_val_score'] = std_score
            records.append(record)

            if mean_score > self.best_score_:
                self.best_score_ = mean_score
                self.best_params_ = dict(params)

        self.cv_results_ = pd.DataFrame(records).sort_values(
            'mean_val_score', ascending=False).reset_index(drop=True)
        return self



### 1.6 Nested Cross-Validation

**The problem:** If we tune hyperparameters using k-fold CV and then report the CV score of the best configuration, we are **using the data twice** — both for selection and for evaluation. This produces an optimistic bias.

**Nested CV** solves this with two loops:
- **Outer loop** ($k_{\text{outer}}$ folds): Provides an unbiased estimate of generalization. The test fold is never seen during inner-loop tuning.
- **Inner loop** ($k_{\text{inner}}$ folds on the outer training fold): Selects the best hyperparameters.

The outer loop's mean score estimates how well "the best model for this problem" will generalize to new data.

In [ ]:
def nested_cross_validate(
    estimator: Any,
    param_grid: dict,
    X: np.ndarray,
    y: np.ndarray,
    k_outer: int = 5,
    k_inner: int = 3,
    scoring: str = 'accuracy',
    scale: bool = True,
    random_state: int = SEED,
) -> dict:
    '''Run nested cross-validation for unbiased hyperparameter evaluation.

    Uses an outer loop for unbiased generalization estimation and an inner
    loop for hyperparameter selection via grid search.

    Args:
        estimator: Base sklearn-compatible estimator.
        param_grid: Hyperparameter grid for the inner GridSearchCV.
        X: Feature matrix of shape (n_samples, n_features).
        y: Label array of shape (n_samples,).
        k_outer: Number of folds in the outer evaluation loop.
        k_inner: Number of folds in the inner selection loop.
        scoring: Metric string passed to GridSearchCV and CV functions.
        scale: Whether to StandardScale inside each fold.
        random_state: Seed for reproducible fold assignments.

    Returns:
        Dictionary with keys:
          'outer_val_scores': list of k_outer test-fold scores,
          'best_params_per_fold': list of best param dicts per outer fold,
          'mean_score': float mean of outer_val_scores,
          'std_score': float std of outer_val_scores.
    '''

    outer_splits = stratified_k_fold_indices(
        y, k_outer, shuffle=True, random_state=random_state)

    outer_scores: list = []
    best_params_list: list = []

    for fold_num, (tr_outer, te_outer) in enumerate(outer_splits):
        X_tr_out: np.ndarray = X[tr_outer]
        y_tr_out: np.ndarray = y[tr_outer]
        X_te_out: np.ndarray = X[te_outer]
        y_te_out: np.ndarray = y[te_outer]

        # Inner loop: grid search on the outer training fold
        gs = GridSearchCV(
            copy.deepcopy(estimator),
            param_grid=param_grid,
            k=k_inner,
            scoring=scoring,
            scale=scale,
            random_state=random_state,
        )
        gs.fit(X_tr_out, y_tr_out)

        # Refit best model on entire outer training fold
        best_model = copy.deepcopy(estimator)
        best_model.set_params(**gs.best_params_)

        if scale:
            sc = StandardScaler()
            X_tr_sc: np.ndarray = sc.fit_transform(X_tr_out)
            X_te_sc: np.ndarray = sc.transform(X_te_out)
        else:
            X_tr_sc = X_tr_out
            X_te_sc = X_te_out

        best_model.fit(X_tr_sc, y_tr_out)
        y_pred_te: np.ndarray = best_model.predict(X_te_sc)

        if scoring == 'accuracy':
            score = accuracy_score(y_te_out, y_pred_te)
        else:
            score = f1_score(y_te_out, y_pred_te, average='macro', zero_division=0)

        outer_scores.append(float(score))
        best_params_list.append(gs.best_params_)
        print(f'  Outer fold {fold_num + 1}/{k_outer}: '
              f'score={score:.4f}  best={gs.best_params_}')

    return {
        'outer_val_scores': outer_scores,
        'best_params_per_fold': best_params_list,
        'mean_score': float(np.mean(outer_scores)),
        'std_score': float(np.std(outer_scores)),
    }



---
## Part 2 — Assembling the Evaluation Harness

The individual functions from Part 1 (CV splitters, `cross_validate_scratch`, `GridSearchCV`, `RandomSearchCV`, `nested_cross_validate`) together form a complete evaluation and hyperparameter-tuning harness. Below we run a sanity check confirming all components work end-to-end before applying them to the Digits dataset in Part 3.

In [ ]:
# ── Sanity check: k-fold index coverage ───────────────────────────────────────
print('=== Sanity Checks ===')
n_test_sanity: int = 100
folds_sanity = k_fold_indices(n_test_sanity, 5)

# Every index appears exactly once in a val fold
all_val_idx: list = []
for _, vl in folds_sanity:
    all_val_idx.extend(vl.tolist())
all_val_sorted = sorted(all_val_idx)
assert all_val_sorted == list(range(n_test_sanity)), \
    'k-fold: each index should appear in exactly one val fold!'
print('k-fold: each sample appears in exactly one val fold ✓')

# Stratified k-fold preserves class proportions within ±1 sample
y_sanity: np.ndarray = np.array([0] * 50 + [1] * 30 + [2] * 20)
folds_strat = stratified_k_fold_indices(y_sanity, 5)
for fold_i, (_, vl) in enumerate(folds_strat):
    c0 = int((y_sanity[vl] == 0).sum())
    c1 = int((y_sanity[vl] == 1).sum())
    c2 = int((y_sanity[vl] == 2).sum())
    # Class 0: ~10/fold, class 1: ~6/fold, class 2: ~4/fold
    assert abs(c0 / len(vl) - 0.50) < 0.12, f'Fold {fold_i}: class 0 ratio off'
print('Stratified k-fold: class proportions preserved per fold ✓')

# cross_validate_scratch returns correct keys
lr_sanity = LogisticRegression(max_iter=200, random_state=SEED)
folds_3 = stratified_k_fold_indices(y_dev[:200], 3)
cv_result_sanity = cross_validate_scratch(lr_sanity, X_dev[:200], y_dev[:200], folds_3)
assert len(cv_result_sanity['val_scores']) == 3, 'Expected 3 val scores'
print(f'cross_validate_scratch: 3-fold sanity accuracy = '
      f'{np.mean(cv_result_sanity["val_scores"]):.4f} ✓')

---
## Part 3 — Applying to Digits: CV, Grid Search, and Nested CV

We now apply all three tools to the Digits development set. We never touch `X_test` / `y_test` until the final estimate in Part 4.

In [ ]:
# ── K-fold vs Stratified K-fold comparison ────────────────────────────────────
print('=== K-Fold vs Stratified K-Fold on Digits (Logistic Regression) ===')

lr_base = LogisticRegression(C=1.0, max_iter=LR_MAX_ITER, random_state=SEED)

folds_kf5 = k_fold_indices(len(y_dev), K_FOLDS)
folds_skf5 = stratified_k_fold_indices(y_dev, K_FOLDS)

cv_kf = cross_validate_scratch(lr_base, X_dev, y_dev, folds_kf5)
cv_skf = cross_validate_scratch(lr_base, X_dev, y_dev, folds_skf5)

print(f'{"Method":<22} {"Fold Scores":<45} {"Mean":>6} {"Std":>7}')
print('-' * 83)
for name, cv_res in [('K-Fold', cv_kf), ('Stratified K-Fold', cv_skf)]:
    scores_str = '  '.join(f'{s:.4f}' for s in cv_res['val_scores'])
    mean_s = float(np.mean(cv_res['val_scores']))
    std_s = float(np.std(cv_res['val_scores']))
    print(f'{name:<22} {scores_str:<45} {mean_s:>6.4f} {std_s:>7.4f}')

# Visualise fold-level score distribution
fig, ax = plt.subplots(figsize=(8, 4))
x_pos = np.arange(K_FOLDS)
width = 0.35
ax.bar(x_pos - width / 2, cv_kf['val_scores'], width, label='K-Fold',
       color='steelblue', edgecolor='white')
ax.bar(x_pos + width / 2, cv_skf['val_scores'], width, label='Stratified K-Fold',
       color='salmon', edgecolor='white')
ax.axhline(np.mean(cv_kf['val_scores']), color='steelblue', ls='--', lw=1.5)
ax.axhline(np.mean(cv_skf['val_scores']), color='salmon', ls='--', lw=1.5)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Fold {i+1}' for i in range(K_FOLDS)])
ax.set_ylabel('Accuracy')
ax.set_title('K-Fold vs Stratified K-Fold: Per-Fold Accuracy')
ax.legend()
ax.set_ylim(0.9, 1.0)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('04_02_kfold_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Grid search for Logistic Regression hyperparameters ───────────────────────
print('=== Grid Search: Logistic Regression on Digits ===')

lr_for_search = LogisticRegression(max_iter=LR_MAX_ITER, random_state=SEED)
param_grid_lr: dict = {
    'C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'solver': ['lbfgs', 'saga'],
}
n_configs_grid: int = len(param_grid_lr['C']) * len(param_grid_lr['solver'])
print(f'Grid size: {n_configs_grid} configurations '
      f'({len(param_grid_lr["C"])} C values × {len(param_grid_lr["solver"])} solvers)')

t0 = time.perf_counter()
gs_lr = GridSearchCV(
    lr_for_search, param_grid_lr,
    k=K_FOLDS, scoring='accuracy', scale=True,
)
gs_lr.fit(X_dev, y_dev)
grid_time: float = time.perf_counter() - t0

print(f'\nGrid search completed in {grid_time:.1f}s')
print(f'Best params : {gs_lr.best_params_}')
print(f'Best CV acc : {gs_lr.best_score_:.4f}')
print()
print('Top-5 configurations:')
display_cols = list(param_grid_lr.keys()) + ['mean_val_score', 'std_val_score']
print(gs_lr.cv_results_[display_cols].head(5).round(4).to_string(index=False))

In [ ]:
# ── Random search for Logistic Regression ─────────────────────────────────────
print('=== Random Search: Logistic Regression on Digits ===')

param_dist_lr: dict = {
    'C': (1e-3, 1e3, 'log'),   # Log-uniform in [0.001, 1000]
}
print(f'Search space: C drawn log-uniformly from [1e-3, 1e3]')
print(f'Budget: {N_ITER_RANDOM} iterations (vs {n_configs_grid} grid configs × {K_FOLDS} folds)')

lr_for_rsearch = LogisticRegression(
    solver='lbfgs', max_iter=LR_MAX_ITER, random_state=SEED)

t0 = time.perf_counter()
rs_lr = RandomSearchCV(
    lr_for_rsearch, param_dist_lr,
    n_iter=N_ITER_RANDOM, k=K_FOLDS,
    scoring='accuracy', scale=True,
)
rs_lr.fit(X_dev, y_dev)
random_time: float = time.perf_counter() - t0

print(f'\nRandom search completed in {random_time:.1f}s')
print(f'Best params : {rs_lr.best_params_}')
print(f'Best CV acc : {rs_lr.best_score_:.4f}')
print()
print('Top-5 configurations:')
print(rs_lr.cv_results_[['C', 'mean_val_score', 'std_val_score']].head(5).round(5).to_string(index=False))

In [ ]:
# ── Nested cross-validation: unbiased generalization estimate ─────────────────
print('=== Nested CV: Unbiased Generalization Estimate ===')
print(f'Outer: {K_OUTER}-fold   Inner: {K_INNER}-fold grid search')
print()

nested_param_grid: dict = {'C': [0.1, 1.0, 10.0]}
lr_nested = LogisticRegression(
    solver='lbfgs', max_iter=LR_MAX_ITER, random_state=SEED)

t0 = time.perf_counter()
nested_result = nested_cross_validate(
    lr_nested, nested_param_grid,
    X_dev, y_dev,
    k_outer=K_OUTER, k_inner=K_INNER,
    scoring='accuracy', scale=True,
)
nested_time: float = time.perf_counter() - t0

print(f'\nNested CV completed in {nested_time:.1f}s')
print(f'Outer scores  : {[round(s, 4) for s in nested_result["outer_val_scores"]]}')
print(f'Mean ± Std    : {nested_result["mean_score"]:.4f} ± {nested_result["std_score"]:.4f}')
print()
print('Best hyperparameters selected per outer fold:')
for i, params in enumerate(nested_result['best_params_per_fold']):
    print(f'  Fold {i+1}: {params}')

# Compare: simple CV score of best config vs nested CV estimate
print(f'\nSimple 5-fold CV (best C={gs_lr.best_params_["C"]}): {gs_lr.best_score_:.4f}')
print(f'Nested CV estimate                               : {nested_result["mean_score"]:.4f}')
bias_est: float = gs_lr.best_score_ - nested_result['mean_score']
print(f'Optimistic bias from non-nested CV               : {bias_est:+.4f}')

---
## Part 4 — Analysis: Learning Curves, Search Efficiency & Validation

We now analyse the CV results in depth:
1. Plot learning curves for Logistic Regression (high-C) and low-C underfitting vs RF overfitting
2. Compare grid vs random search efficiency
3. Validate our CV scores against sklearn
4. Evaluate the final selected model on the held-out test set

In [ ]:
# ── Learning curves for LR and RF ─────────────────────────────────────────────
print('Computing learning curves (this may take ~30s)...')

max_train: int = len(y_dev) - len(y_dev) // K_FOLDS  # max usable training size
train_sizes: np.ndarray = np.linspace(
    n_classes * 5, max_train, num=N_TRAIN_SIZES, dtype=int)

# Logistic Regression: well-regularised (C=1) vs over-regularised (C=0.001)
lr_good = LogisticRegression(C=1.0, max_iter=LR_MAX_ITER, random_state=SEED)
lr_underfit = LogisticRegression(C=0.001, max_iter=LR_MAX_ITER, random_state=SEED)
rf_overfit = RandomForestClassifier(
    n_estimators=RF_N_EST, max_depth=None, random_state=SEED)

lc_lr_good = learning_curve_scratch(lr_good, X_dev, y_dev, train_sizes, k=3)
print('  LR (C=1.0) done.')
lc_lr_under = learning_curve_scratch(lr_underfit, X_dev, y_dev, train_sizes, k=3)
print('  LR (C=0.001) done.')
lc_rf = learning_curve_scratch(rf_overfit, X_dev, y_dev, train_sizes, k=3, scale=False)
print('  RF (unlimited depth) done.')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_learning_curve(lc_lr_good, title='LR C=1 (Well-tuned)', ax=axes[0])
plot_learning_curve(lc_lr_under, title='LR C=0.001 (High Bias)', ax=axes[1])
plot_learning_curve(lc_rf, title='RF (Unlimited Depth)', ax=axes[2])

plt.tight_layout()
plt.savefig('04_02_learning_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nLearning curve diagnosis:')
gap_good  = lc_lr_good['train_mean'][-1]  - lc_lr_good['val_mean'][-1]
gap_under = lc_lr_under['train_mean'][-1] - lc_lr_under['val_mean'][-1]
gap_rf    = lc_rf['train_mean'][-1]       - lc_rf['val_mean'][-1]
print(f'  LR C=1   : train-val gap = {gap_good:.4f}  → well-balanced')
print(f'  LR C=0.001: train-val gap = {gap_under:.4f}  → small gap but low scores = HIGH BIAS')
print(f'  RF (deep) : train-val gap = {gap_rf:.4f}  → large gap = HIGH VARIANCE')

In [ ]:
# ── Grid vs Random search: coverage of the hyperparameter space ───────────────
print('=== Grid vs Random Search Coverage Comparison ===')

# Simulate a 2D hyperparameter space: C and gamma
# Grid search: 5 C values × 5 gamma values = 25 evaluations
# Random search: 25 evaluations drawn uniformly from [log(1e-3), log(1e3)]^2
rng_cov = np.random.RandomState(SEED)
n_evals: int = 25

# Grid points
grid_c = np.logspace(-2, 2, 5)
grid_gamma = np.logspace(-2, 2, 5)
grid_c_mesh, grid_g_mesh = np.meshgrid(grid_c, grid_gamma)
grid_pts_c = grid_c_mesh.flatten()
grid_pts_g = grid_g_mesh.flatten()

# Random points
random_pts_c = np.exp(rng_cov.uniform(np.log(1e-2), np.log(1e2), size=n_evals))
random_pts_g = np.exp(rng_cov.uniform(np.log(1e-2), np.log(1e2), size=n_evals))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.scatter(np.log10(grid_pts_c), np.log10(grid_pts_g),
           s=80, c='steelblue', marker='s', label=f'Grid ({n_evals} pts)')
ax.set_xlabel('log10(C)')
ax.set_ylabel('log10(gamma)')
ax.set_title('Grid Search: Only 5 Distinct C Values')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.scatter(np.log10(random_pts_c), np.log10(random_pts_g),
           s=80, c='salmon', marker='o', label=f'Random ({n_evals} pts)')
ax.set_xlabel('log10(C)')
ax.set_ylabel('log10(gamma)')
ax.set_title('Random Search: Explores Full Range of Both Axes')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(
    'Grid vs Random Search Coverage\n'
    '(If only C matters, random covers 25 distinct C values; grid covers only 5)',
    fontsize=10,
)
plt.tight_layout()
plt.savefig('04_02_search_coverage.png', dpi=100, bbox_inches='tight')
plt.show()

n_unique_c_grid: int = len(np.unique(grid_pts_c))
n_unique_c_rand: int = len(np.unique(np.round(random_pts_c, 3)))
print(f'Unique C values explored — Grid: {n_unique_c_grid}, Random: {n_unique_c_rand}')
print('Key insight: with the same budget, random search explores each axis more thoroughly.')
print('Benefit grows with the number of dimensions — grid search scales exponentially.')

In [ ]:
# ── CV variance vs k: how does fold count affect estimate quality? ────────────
print('=== Effect of k on CV Estimate Variance ===')
print('Computing cross-validation for k = 2, 3, 5, 10, 20 (this may take ~30s)...')

k_values: list = [2, 3, 5, 10, 20]
n_repeats: int = 10   # Repeat each k-fold with different seeds to measure variance

lr_kvar = LogisticRegression(C=1.0, max_iter=LR_MAX_ITER, random_state=SEED)

k_means: list = []
k_stds: list = []
k_mean_of_stds: list = []   # Average within-repetition std

for k_val in k_values:
    rep_means: list = []
    rep_stds: list = []
    for rep in range(n_repeats):
        folds_k = stratified_k_fold_indices(y_dev, k_val, shuffle=True,
                                             random_state=SEED + rep)
        cv_k = cross_validate_scratch(lr_kvar, X_dev, y_dev, folds_k, scale=True)
        rep_means.append(float(np.mean(cv_k['val_scores'])))
        rep_stds.append(float(np.std(cv_k['val_scores'])))

    k_means.append(float(np.mean(rep_means)))        # mean of means across repetitions
    k_stds.append(float(np.std(rep_means)))           # std of the MEAN across repetitions
    k_mean_of_stds.append(float(np.mean(rep_stds)))  # avg within-run fold std

print(f'\n{"k":>4} {"Mean acc":>10} {"Std of mean":>13} {"Avg fold std":>14}')
print('-' * 45)
for k_val, mn, std_mn, avg_fd_std in zip(k_values, k_means, k_stds, k_mean_of_stds):
    print(f'{k_val:>4} {mn:>10.4f} {std_mn:>13.5f} {avg_fd_std:>14.5f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.errorbar(k_values, k_means, yerr=k_stds, fmt='bo-', lw=2, ms=8,
            capsize=5, label='Mean ± std(mean)')
ax.set_xlabel('Number of folds k')
ax.set_ylabel('Estimated accuracy')
ax.set_title('CV Mean Accuracy vs k')
ax.set_xticks(k_values)
ax.grid(True, alpha=0.3)
ax.legend()

ax = axes[1]
ax.plot(k_values, k_stds, 'bo-', lw=2, ms=8, label='Std of CV mean (across repeats)')
ax.plot(k_values, k_mean_of_stds, 'rs-', lw=2, ms=8, label='Avg within-run fold std')
ax.set_xlabel('Number of folds k')
ax.set_ylabel('Standard deviation')
ax.set_title('CV Variance vs k (lower is more reliable)')
ax.set_xticks(k_values)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('How k Affects Cross-Validation Estimate Stability', fontsize=11)
plt.tight_layout()
plt.savefig('04_02_cv_variance_vs_k.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nKey insight: increasing k reduces both bias and variance of the CV estimate,')
print('but the benefit diminishes above k=5 while compute cost grows linearly with k.')
print('k=5 and k=10 are standard choices offering a good bias-variance tradeoff.')


In [ ]:
# ── Random search convergence: best score vs budget ───────────────────────────
print('=== Random Search Convergence: Best Score vs Number of Iterations ===')

# Run random search with increasing budgets and record best score at each step
budgets: list = list(range(1, N_ITER_RANDOM + 1))
best_at_budget: list = []
current_best: float = -1.0
rng_conv = np.random.RandomState(SEED)
lr_conv = LogisticRegression(solver='lbfgs', max_iter=LR_MAX_ITER, random_state=SEED)

# Use pre-computed splits for reproducibility
splits_conv = stratified_k_fold_indices(y_dev, K_FOLDS, shuffle=True, random_state=SEED)

for iteration in range(N_ITER_RANDOM):
    c_sampled: float = float(np.exp(rng_conv.uniform(np.log(1e-3), np.log(1e3))))
    model_conv = copy.deepcopy(lr_conv)
    model_conv.set_params(C=c_sampled)
    cv_conv = cross_validate_scratch(model_conv, X_dev, y_dev, splits_conv, scale=True)
    mean_cv: float = float(np.mean(cv_conv['val_scores']))
    if mean_cv > current_best:
        current_best = mean_cv
    best_at_budget.append(current_best)

# Also compute grid search best at each budget (sorted by score)
if gs_lr.cv_results_ is not None:
    grid_scores_sorted: np.ndarray = gs_lr.cv_results_['mean_val_score'].values
    # Grid order is arbitrary — show cumulative best
    grid_cumulative_best: list = []
    grid_best: float = -1.0
    for s in grid_scores_sorted:
        if float(s) > grid_best:
            grid_best = float(s)
        grid_cumulative_best.append(grid_best)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(budgets, best_at_budget, 'b-o', lw=2, ms=5, label='Random search best')
if gs_lr.cv_results_ is not None:
    n_grid_configs = len(grid_cumulative_best)
    ax.axhline(grid_cumulative_best[-1], color='red', ls='--', lw=2,
               label=f'Grid best ({n_grid_configs} configs)')
ax.set_xlabel('Number of evaluations (budget)')
ax.set_ylabel('Best CV accuracy found so far')
ax.set_title('Random Search Convergence')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Show the sampled C values colored by their CV score
c_vals_track: list = []
score_vals_track: list = []
rng_track = np.random.RandomState(SEED)  # Re-sample same sequence
for _ in range(N_ITER_RANDOM):
    c_t = float(np.exp(rng_track.uniform(np.log(1e-3), np.log(1e3))))
    c_vals_track.append(c_t)

ax = axes[1]
sc_plot = ax.scatter(
    np.log10(np.array(c_vals_track)),
    rs_lr.cv_results_['mean_val_score'].values[:N_ITER_RANDOM],
    c=rs_lr.cv_results_['mean_val_score'].values[:N_ITER_RANDOM],
    cmap='viridis', s=60, alpha=0.8, edgecolors='none',
)
plt.colorbar(sc_plot, ax=ax, label='CV accuracy')
ax.axvline(np.log10(rs_lr.best_params_['C']), color='red', ls='--', lw=2,
           label=f'Best C={rs_lr.best_params_["C"]:.4f}')
ax.set_xlabel('log10(C)')
ax.set_ylabel('CV accuracy')
ax.set_title('Random Search: Sampled C Values vs Score')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Random Search Behavior', fontsize=11)
plt.tight_layout()
plt.savefig('04_02_random_search_convergence.png', dpi=100, bbox_inches='tight')
plt.show()

# How many random iterations to reach 99% of the grid best?
threshold_99: float = grid_cumulative_best[-1] * 0.9999
budget_to_99: int = next(
    (i + 1 for i, s in enumerate(best_at_budget) if s >= threshold_99),
    N_ITER_RANDOM,
)
print(f'Grid best CV score (all {n_grid_configs} configs) : {grid_cumulative_best[-1]:.4f}')
print(f'Random search best ({N_ITER_RANDOM} samples)         : {best_at_budget[-1]:.4f}')
print(f'Random reached >=99.99%% of grid best after {budget_to_99} evaluations.')
print('Random search finds near-optimal configurations with a fraction of the full grid budget.')


In [ ]:
# ── Validate our CV against sklearn's cross_val_score ─────────────────────────
print('=== Scratch CV vs sklearn cross_val_score Validation ===')

lr_val = LogisticRegression(C=1.0, max_iter=LR_MAX_ITER, random_state=SEED)

# Our implementation
folds_val = stratified_k_fold_indices(y_dev, K_FOLDS)
cv_ours = cross_validate_scratch(lr_val, X_dev, y_dev, folds_val, scale=True)
mean_ours: float = float(np.mean(cv_ours['val_scores']))
std_ours: float = float(np.std(cv_ours['val_scores']))

# sklearn's implementation (note: sklearn StratifiedKFold uses a different default shuffle)
lr_pipeline = Pipeline([('sc', StandardScaler()),
                         ('lr', LogisticRegression(C=1.0, max_iter=LR_MAX_ITER,
                                                   random_state=SEED))])
skf_sk = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
scores_sk: np.ndarray = cross_val_score(lr_pipeline, X_dev, y_dev,
                                         cv=skf_sk, scoring='accuracy')
mean_sk: float = float(scores_sk.mean())
std_sk: float = float(scores_sk.std())

print(f'Our implementation : {mean_ours:.4f} ± {std_ours:.4f}')
print(f'sklearn            : {mean_sk:.4f} ± {std_sk:.4f}')
print(f'Difference in mean : {abs(mean_ours - mean_sk):.4f}')
print()
print('Note: small differences may arise from different internal shuffle seeds.')
print('Both should be well within 1 std of each other.')
assert abs(mean_ours - mean_sk) < 0.02, 'CV mean diverges too much from sklearn!'
print('Validation passed ✓')

In [ ]:
# ── Final test set evaluation ──────────────────────────────────────────────────
# Apply the best configuration found via nested CV to the held-out test set

print('=== Final Test Set Evaluation (using best nested CV configuration) ===')
print()

# Majority vote among best params per fold
best_c_counts: dict = {}
for bp in nested_result['best_params_per_fold']:
    c_val = bp['C']
    best_c_counts[c_val] = best_c_counts.get(c_val, 0) + 1
final_c: float = max(best_c_counts, key=lambda k: best_c_counts[k])
print(f'C selection votes across outer folds: {best_c_counts}')
print(f'Final C selected by majority vote    : {final_c}')

# Refit on entire dev set with best C
sc_final = StandardScaler()
X_dev_sc: np.ndarray = sc_final.fit_transform(X_dev)
X_test_sc: np.ndarray = sc_final.transform(X_test)

final_model = LogisticRegression(
    C=final_c, solver='lbfgs', max_iter=LR_MAX_ITER, random_state=SEED)
final_model.fit(X_dev_sc, y_dev)

y_pred_final: np.ndarray = final_model.predict(X_test_sc)
test_acc: float = accuracy_score(y_test, y_pred_final)

print(f'\nNested CV estimate  : {nested_result["mean_score"]:.4f} ± {nested_result["std_score"]:.4f}')
print(f'Final test accuracy : {test_acc:.4f}')
print(f'Estimation error    : {abs(test_acc - nested_result["mean_score"]):+.4f}')
print()
print('The nested CV estimate is an honest predictor of held-out test performance.')
print('Non-nested CV score was', f'{gs_lr.best_score_:.4f},',
      f'overestimating by {gs_lr.best_score_ - test_acc:+.4f}.')

# Summary comparison table
summary: pd.DataFrame = pd.DataFrame([
    {'Method': 'Simple K-Fold CV (best config)', 'Score': gs_lr.best_score_,
     'Note': 'Optimistically biased'},
    {'Method': f'Nested CV ({K_OUTER}×{K_INNER})',
     'Score': nested_result['mean_score'],
     'Note': 'Unbiased generalization estimate'},
    {'Method': 'True Test Set', 'Score': test_acc,
     'Note': 'Ground truth'},
]).set_index('Method')
print('\nFinal comparison:')
print(summary.round(4).to_string())

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **K-fold CV is a biased-but-low-variance estimator of generalization.** With $k=5$ or $k=10$, the variance across folds is small enough to compare models reliably. Stratified k-fold is always preferable when class imbalance exists — it guarantees each fold reflects the true class distribution.

2. **Simple CV of a tuned model is optimistically biased.** Selecting hyperparameters based on CV score and then reporting that same CV score double-dips on the data. Nested CV separates the tuning and evaluation phases and produces a nearly unbiased generalization estimate.

3. **Random search beats grid search under a fixed budget.** For $n$ evaluations in a $d$-dimensional space, grid search explores only $n^{1/d}$ values per axis, while random search explores $n$ distinct values per axis. As $d$ grows, random search's advantage becomes decisive.

4. **Learning curves diagnose bias and variance.** High bias: both train and val scores are low with a small gap — adding data barely helps. High variance: train score is high but val score is much lower — adding data closes the gap. The gap between curves is the key diagnostic signal.

5. **The test set is sacred.** It must be separated before any CV or tuning, and inspected exactly once for the final reported result. Repeated evaluation on the test set introduces test-set leakage, which is the same optimistic bias as non-nested CV.

### What's Next

→ **04-03 (Feature Engineering & Pipelines)** builds on the sklearn Pipeline pattern introduced here to construct reproducible preprocessing workflows that prevent data leakage at scale.